In [ ]:
import tomllib
from pathlib import Path

import pandas as pd
import yfinance as yf

### Check Tickers from portfolio.toml


In [ ]:
root = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
with open(root / "config" / "portfolio.toml", "rb") as f:
    cfg = tomllib.load(f)

expected_ccy = cfg["portfolio"]["currency"]

rows = []
for asset in cfg["portfolio"]["assets"]:
    row = {"name": asset["name"], "ticker": asset["ticker"]}
    try:
        t = yf.Ticker(asset["ticker"])
        hist = t.history(period="max")
        currency = t.info.get("currency")
        row.update(
            rows=len(hist),
            first=hist.index.min().date() if not hist.empty else None,
            last=hist.index.max().date() if not hist.empty else None,
            currency=currency,
            ok=not hist.empty and currency == expected_ccy,
        )
    except Exception as e:
        row.update(ok=False, error=str(e))
    rows.append(row)

report = pd.DataFrame(rows)
display(report)

for _, r in report[~report["ok"]].iterrows():
    print(f"WARNING: {r['ticker']} failed (no data, currency != {expected_ccy}, or error)")